# Supervised vs. Unsupervised Learning: Iris Demo

This short demo uses the classic Iris flower dataset to show the difference between two common machine learning tasks. It is intentionally smaller than the support-ticket model so we can focus on the idea, not the size of the data.

- **Supervised learning** trains with examples that include the correct answer.
- **Unsupervised learning** trains with examples that do not include the correct answer.

We will use the same flower measurements in both sections so the only major difference is whether the model receives labels during training.

A useful mental model:

- Supervised learning is like studying with an answer key.
- Unsupervised learning is like sorting a pile of examples into groups without being told the official categories.

Both are machine learning, but they answer different kinds of questions.

## What We Are Learning

The Iris dataset contains measurements for 150 flowers. Each flower has four input measurements:

- sepal length
- sepal width
- petal length
- petal width

Each flower also has a known species: `setosa`, `versicolor`, or `virginica`.

In the supervised section, the species is the answer the model learns to predict. In the unsupervised section, we hide that answer from the model and ask it to discover groups from the measurements alone.

The same row of data can be used in both ways:

| Measurement columns | Species column | How we use it |
|---|---|---|
| Used as inputs | Used as the answer | Supervised learning |
| Used as inputs | Hidden from the model | Unsupervised learning |

That is the core distinction. It is not about whether the dataset is simple or complex. It is about whether the training process includes known answers.

In [ ]:
import os
import warnings

# This tiny demo does not need parallel workers. Keeping joblib to one worker
# also avoids a noisy Windows physical-core detection warning on some systems.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings(
    "ignore",
    message="Could not find the number of physical cores*",
    category=UserWarning,
    module="joblib.externals.loky.backend.context",
)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 20)

RANDOM_STATE = 42

print("Setup complete. RANDOM_STATE=42 makes the demo repeatable.")

The setup cell imports the tools we need:

- `pandas` stores the dataset in tables.
- `matplotlib` creates simple charts.
- `LogisticRegression` is the supervised classifier.
- `KMeans` is the unsupervised clustering algorithm.
- The metrics help us evaluate what the models did.

`RANDOM_STATE` is a repeatability seed. The number `42` is not mathematically special. It just makes random steps, such as train/test splitting and cluster initialization, repeatable for the class.

## Load the Dataset

`scikit-learn` includes the Iris dataset, so we do not need to download a file.

The important teaching point is the split between:

- **features**, also called inputs or `X`
- **labels**, also called answers, targets, or `y`

Machine learning code often uses `X` for the input columns and `y` for the answer column.

This is more than a naming convention. It helps us separate what the model is allowed to know from what we want it to learn.

- `X` answers the question: what information will be available when we need a prediction or grouping?
- `y` answers the question: what known answer are we trying to predict?

For supervised learning, we train with both `X` and `y`. For unsupervised learning, we train with `X` only.

In [ ]:
iris = load_iris(as_frame=True)

X = iris.data
y = iris.target
target_names = [str(name) for name in iris.target_names]

iris_table = X.copy()
iris_table["species"] = y.map(lambda index: target_names[index])

print(f"Rows: {iris_table.shape[0]}")
print(f"Input columns: {list(X.columns)}")
print(f"Species labels: {target_names}")

iris_table.head()

In [ ]:
iris_table["species"].value_counts()

The species counts show that the dataset is balanced: each species has the same number of rows.

That makes this a good teaching dataset. If one species had many more examples than the others, accuracy could be misleading because a model might do well by favoring the majority class.

For the supervised section, these labels are the answers. For the unsupervised section, we keep the labels only so we can discuss the results after clustering.

## Part 1: Supervised Learning

In supervised learning, the training examples include the correct answer. The model can compare its predictions to those answers while it learns.

For this section:

- **Inputs:** flower measurements
- **Answer:** species
- **Task:** learn to predict species for a flower the model has not seen before

This is similar to the support-ticket classifier in the main workshop. There, the input is ticket text and the answer is the support queue. Here, the input is flower measurements and the answer is species.

Supervised learning is appropriate when you have historical examples with trusted labels and you want the model to make that same kind of decision for future examples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")

The train/test split gives the model some rows to learn from and saves other rows for evaluation.

This matters because we do not only want to know whether the model can memorize examples it already saw. We want to estimate how it behaves on new examples.

In this cell:

- `X_train` and `y_train` are the examples and answers used for training.
- `X_test` is held back until after training.
- `y_test` is also held back so we can check whether the model's predictions were right.
- `stratify=y` keeps the species mix similar in the training and test sets.

The model does not see `X_test` or `y_test` while it trains.

In [ ]:
supervised_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
supervised_model.fit(X_train, y_train)

predictions = supervised_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy on held-out test flowers: {accuracy:.3f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=target_names))

This cell does three things.

First, it creates a `LogisticRegression` model. Logistic Regression is a standard supervised classification algorithm. It is a reasonable first model here because the goal is to choose one category from a small set of known categories.

Second, `.fit(X_train, y_train)` trains the model. This is the supervised part: the model receives both the measurements and the correct species labels.

Third, `.predict(X_test)` asks the model to predict species for flowers it did not train on. We then compare those predictions to `y_test`, the held-back answer key.

The classification report gives more detail than accuracy alone:

- **precision** asks: when the model predicted this species, how often was it right?
- **recall** asks: of the real flowers in this species, how many did the model find?
- **f1-score** combines precision and recall into one score.
- **support** is the number of test examples for that species.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=target_names,
    cmap="Blues",
    xticks_rotation=35,
    ax=ax,
    colorbar=False,
)
ax.set_title("Supervised Learning: Predicted Species")
plt.tight_layout()
plt.show()

### How to Read the Confusion Matrix

A confusion matrix compares the true answers to the model's predictions.

- Rows are the real species.
- Columns are the predicted species.
- Numbers on the diagonal are correct predictions.
- Numbers away from the diagonal are mistakes.

This evaluation is possible because supervised learning has known answers for the test set.

That is one practical advantage of supervised learning. If we have trustworthy labels, we can measure performance directly. We can say not only that the model made predictions, but how often those predictions matched known correct answers.

In a real project, this is also where we would ask whether the errors are acceptable. A high accuracy score is helpful, but the specific mistakes may matter more than the average.

## Part 2: Unsupervised Learning

In unsupervised learning, the model does not receive the answer column during training. It only receives the input measurements and searches for patterns in those measurements.

For this section:

- **Inputs:** flower measurements
- **Answer given during training:** none
- **Task:** discover groups that appear naturally in the measurements

We will use KMeans clustering. KMeans tries to place rows into a fixed number of groups, called clusters. We will ask for 3 clusters because we know this dataset has 3 species, but KMeans will not be told which flower belongs to which species.

KMeans works roughly like this:

- Choose a number of clusters.
- Place temporary cluster centers in the data.
- Assign each row to the nearest center.
- Move the centers to better match the assigned rows.
- Repeat until the assignments settle down.

Notice what is missing: there is no answer key. KMeans does not know the words `setosa`, `versicolor`, or `virginica`. It only knows that some flowers have similar measurements.

In [ ]:
unsupervised_model = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
clusters = unsupervised_model.fit_predict(X)

cluster_table = iris_table.copy()
cluster_table["cluster"] = clusters

cluster_table.head()

The `cluster` column is not a species name. It is just the group number KMeans assigned.

Cluster numbers are arbitrary. Cluster `0` does not mean better, first, or setosa. It only means KMeans found one group and named it `0`.

This is a major difference from supervised classification. The supervised model outputs labels from the answer column because those labels were part of training. The unsupervised model outputs group IDs because it only learned a grouping structure.

In [ ]:
print("Cluster sizes:")
display(cluster_table["cluster"].value_counts().sort_index())

print("\nCompare discovered clusters to the real species labels:")
pd.crosstab(cluster_table["cluster"], cluster_table["species"])

The table above is a teaching aid. It compares the discovered clusters to the real species labels after KMeans has already finished.

This is different from training with labels. During training, KMeans did not know whether a row was setosa, versicolor, or virginica. It only grouped rows by measurement similarity.

If a cluster mostly contains one species, that means the measurements contain structure that lines up with the known species. If a cluster mixes multiple species, that means those species overlap in the measurement space the model used.

In [ ]:
agreement = adjusted_rand_score(y, clusters)

print(f"Cluster/species agreement score: {agreement:.3f}")
print("A score near 1.0 means the discovered groups line up well with the known labels.")
print("A score near 0.0 means the grouping is no better than random.")

The agreement score is not the same as supervised accuracy.

Supervised accuracy asks, `Did the model predict the correct label?` That question makes sense because the supervised model was trained to predict labels.

The cluster agreement score asks a different question: `Do the discovered groups line up with the labels we know after the fact?` That is useful here because Iris has labels, but many unsupervised projects do not have a reliable answer key.

We are allowed to compare clusters to species after clustering because this is a teaching dataset.

That comparison helps us understand what the algorithm discovered. But the key point remains: KMeans did not use species labels while it was learning.

The scatter plot below shows the discovered clusters using only two of the four measurements: petal length and petal width.

The model actually trained on all four measurement columns. We plot two columns because two-dimensional charts are easier to read.

Each point is one flower. The color is the cluster assignment, not the true species label.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    X["petal length (cm)"],
    X["petal width (cm)"],
    c=clusters,
    cmap="viridis",
    edgecolor="black",
    linewidth=0.4,
)
ax.set_title("Unsupervised Learning: KMeans Clusters")
ax.set_xlabel("Petal length (cm)")
ax.set_ylabel("Petal width (cm)")
legend = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend)
plt.tight_layout()
plt.show()

## Predicting or Grouping a New Flower

A supervised classifier can predict a specific species label because it learned from species labels.

An unsupervised clustering model can assign the flower to a discovered group, but that group is not automatically a species name.

This is often the easiest way to remember the difference:

- A supervised model can say, `I think this is class B`, because class B was part of training.
- An unsupervised model can say, `This looks like group 2`, because it learned groups, not class names.

You may later decide that group 2 mostly corresponds to a real-world category, but that interpretation comes from human analysis after clustering.

In [ ]:
new_flower = pd.DataFrame(
    [
        {
            "sepal length (cm)": 5.8,
            "sepal width (cm)": 2.7,
            "petal length (cm)": 4.1,
            "petal width (cm)": 1.0,
        }
    ]
)

species_prediction = supervised_model.predict(new_flower)[0]
cluster_assignment = unsupervised_model.predict(new_flower)[0]

print("New flower measurements:")
display(new_flower)

print(f"Supervised prediction: {target_names[species_prediction]}")
print(f"Unsupervised cluster assignment: cluster {cluster_assignment}")

## Recap

| Question | Supervised learning | Unsupervised learning |
|---|---|---|
| Does training use known answers? | Yes | No |
| What did we provide as input? | Flower measurements | Flower measurements |
| What did the model learn? | A mapping from measurements to species | Groups based on measurement similarity |
| What is the output? | A predicted species label | A cluster number |
| How do we evaluate it here? | Compare predictions to known test labels | Compare discovered clusters to known labels after the fact |
| Best used when... | You have labeled examples and want future predictions | You do not have labels, or you want to explore hidden structure |

The important difference is not the dataset. The important difference is whether the model trains with an answer column.

For the support-ticket classifier, we are doing supervised learning because each historical ticket has a known queue. The model learns from past examples where the answer is already recorded.

An unsupervised support-ticket example would ask a different question. Instead of predicting a known queue, we might ask the model to group tickets into themes so we can discover common issue types, duplicate problems, or emerging support patterns. That could be useful, but it would not automatically produce official queue labels.